# MATLAB ↔ Python cross-language validation

This advanced example compares the **final features** produced by MATLAB and Python from the same tracked synthetic recording. Both languages use their public BioSignalsPlux/OpenSignals HDF5 readers; no physiological signal is regenerated here.

First generate the MATLAB reference from the repository root:

```matlab
addpath(genpath('matlab'))
addpath('examples/cross_language_validation')
export_matlab_reference
```

This writes `validation-output/matlab_features.json`. ECG frequency method 2 is outside this comparison. EDA requires official, unmodified cvxEDA v1.1.0 in both languages. The MATLAB exporter rejects any other `cvxEDA.m`, including locally modified copies that clip negative phasic values.

## 1. Load the common synthetic recording

The tracked fixture contains 180 seconds at 100 Hz. Channels 1, 2, 3, 4, 9, and 10 provide ECG, EDA, RIP, temperature, fNIRS red, and fNIRS infrared respectively.

In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import json
import numpy as np
import pandas as pd

from multimodalphysiokit.io import read_biosignalsplux_hdf5
from multimodalphysiokit.processors import (
    ECGProcessor, EDAProcessor, FNIRSProcessor,
    RespirationProcessor, TemperatureProcessor,
)

fixture_relative_path = Path('examples/sample_data/synthetic_multimodal_180s.hdf5')
repository_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / fixture_relative_path).is_file()),
    None,
)
if repository_root is None:
    raise FileNotFoundError('Could not locate the repository synthetic fixture.')
fixture_path = repository_root / fixture_relative_path

recording = read_biosignalsplux_hdf5(fixture_path, fnirs_column_order=(0, 1))
expected_channels = {
    'ecg': 1, 'eda': 2, 'rip': 3, 'temp': 4,
    'fnirs_red': 9, 'fnirs_infrared': 10,
}
assert recording.signal_names() == list(expected_channels)
assert all(recording.get_signal(name).metadata['channel_id'] == channel
           for name, channel in expected_channels.items())
assert all(recording.get_signal(name).n_samples == 18_000 for name in expected_channels)
assert all(recording.get_signal(name).sampling_frequency == 100 for name in expected_channels)
recording.signal_names()

## 2. Define the three common phases

The half-open intervals `[0,60)`, `[60,120)`, and `[120,180)` map to exactly 6,000 samples each. Python uses zero-based, stop-exclusive slices; the MATLAB exporter verifies the corresponding one-based slices.

In [ ]:
phase_names = ['phase_1', 'phase_2', 'phase_3']
phase_intervals = np.array([[0.0, 60.0, 120.0], [60.0, 120.0, 180.0]])
recording.set_phases(phase_intervals, phase_names)

expected_phase_indices = np.array([[0, 6000, 12000], [6000, 12000, 18000]])
assert np.array_equal(recording.phase_indices('ecg'), expected_phase_indices)
expected_phase_indices

## 3. Process the signals in Python

ECG uses STFT frequency method 1. fNIRS uses age 25 and the phase-1 baseline after trimming 10 seconds at each end.

EDA receives channel 2 from the shared HDF5 file and runs the complete workflow with official, unmodified cvxEDA v1.1.0. cvxEDA is separate GPL-licensed software and is not distributed with MultimodalPhysioKit; working Python and MATLAB installations are required. See the [dependency documentation](../../docs/dependencies.md) for the upstream project and integration options. MultimodalPhysioKit leaves cvxEDA outputs unchanged, then detects responses using filtered-phasic trough-to-peak amplitude with the configurable 0.01 µS minimum. Solver and runtime differences may produce small numerical differences, so continuous final EDA features use tolerance-based comparison; accepted event counts must match exactly. Set `show_eda_diagnostics` below to display filtered EDA, tonic/SCL, and phasic/SCR traces.

In [ ]:
python_features = {}
processor_inputs = {
    'ecg': (ECGProcessor(label_frequency_analysis=1), 'ecg'),
    'respiration': (RespirationProcessor(), 'rip'),
    'temperature': (TemperatureProcessor(), 'temp'),
}
for modality, (processor, signal_name) in processor_inputs.items():
    results = recording.process_phases(processor, signal_name)
    python_features[modality] = {phase: result.features for phase, result in results.items()}

try:
    installed_cvxeda_version = version('cvxeda')
except PackageNotFoundError as error:
    raise RuntimeError(
        'Official external cvxEDA v1.1.0 is required. Install cvxeda==1.1.0 '
        'as described in docs/dependencies.md.'
    ) from error
if installed_cvxeda_version != '1.1.0':
    raise RuntimeError(
        f'cvxeda==1.1.0 is required; found cvxeda=={installed_cvxeda_version}.'
    )
show_eda_diagnostics = False
eda_processor = EDAProcessor(minimum_scr_amplitude=0.01)
eda_results = recording.process_phases(
    eda_processor, 'eda',
    process_kwargs={'show_plots': show_eda_diagnostics},
)
python_features['eda'] = {phase: result.features for phase, result in eda_results.items()}
python_eda_event_counts = {
    phase: round(result.features['eda_scr_per_min'] * 59.99 / 60)
    for phase, result in eda_results.items()
}

fnirs_processor = FNIRSProcessor(age_years=25)
baseline_red_signal = recording.extract_phase('fnirs_red', 'phase_1')
baseline_infrared_signal = recording.extract_phase('fnirs_infrared', 'phase_1')
baseline_red, baseline_infrared = fnirs_processor.compute_baseline_values(
    baseline_red_signal, baseline_infrared_signal
)
baseline_downsampled = fnirs_processor.downsample_currents(
    baseline_red_signal, baseline_infrared_signal
)
assert len(baseline_downsampled['red'][100:-100]) == 400  # [10, 50) s at 10 Hz
fnirs_results = recording.process_phases(
    fnirs_processor, ('fnirs_red', 'fnirs_infrared'),
    process_kwargs={
        'baseline_red_value': baseline_red,
        'baseline_infrared_value': baseline_infrared,
    },
)
python_features['fnirs'] = {phase: result.features for phase, result in fnirs_results.items()}

## 4. Load MATLAB reference features

The MATLAB exporter writes final features only, plus compact metadata confirming the common frequency, phases, channels, age, and baseline convention.

In [ ]:
matlab_path = repository_root / 'validation-output/matlab_features.json'
if not matlab_path.is_file():
    raise FileNotFoundError('Run export_matlab_reference in MATLAB before this section.')
matlab_reference = json.loads(matlab_path.read_text(encoding='utf-8'))

metadata = matlab_reference['metadata']
assert metadata['sampling_frequency'] == 100
assert metadata['samples_per_phase'] == 6000
assert metadata['phases'] == {
    'phase_1': [0, 60], 'phase_2': [60, 120], 'phase_3': [120, 180]
}
assert metadata['fnirs_age'] == 25
assert metadata['fnirs_red_channel'] == 9
assert metadata['fnirs_infrared_channel'] == 10
assert metadata['fnirs_baseline_phase'] == 'phase_1'
assert metadata['fnirs_baseline_interval_seconds'] == [10, 50]
assert metadata['eda_decomposition'] == 'official_cvxEDA_v1.1.0'
assert metadata['minimum_scr_amplitude_us'] == 0.01
matlab_eda_event_counts = matlab_reference['eda_accepted_event_counts']
assert python_eda_event_counts == matlab_eda_event_counts, (
    f'Accepted EDA event counts differ: Python {python_eda_event_counts}; '
    f'MATLAB {matlab_eda_event_counts}'
)

## 5. Prepare feature comparison

Feature names are explicit: 11 ECG, 8 EDA, 14 respiration, 8 temperature, and 38 fNIRS definitions. The helper below applies the unchanged modality tolerances and uses a symmetric relative difference. NaN and infinity fail.

In [ ]:
fnirs_suffixes = [
    'min', 'max', 'mean', 'variance', 'std', 'skewness', 'kurtosis',
    'power_band', 'max_frequency', 'median_frequency', 'spectral_entropy',
    'mean_diff', 'peak_to_peak', 'slope', 'zero_crossing', 'polarity',
    'entropy', 'auc', 'total_energy',
]

expected_features = {
    'ecg': [
        'ecg_bpm_mean', 'ecg_bpm_std', 'ecg_bpm_median', 'ecg_pnn50',
        'ecg_pnn50_max', 'ecg_plf_mean', 'ecg_plf_std', 'ecg_phf_mean',
        'ecg_phf_std', 'ecg_lf_hf_mean', 'ecg_lf_hf_std',
    ],
    'eda': [
        'eda_scl_mean', 'eda_scl_std', 'eda_scl_slope',
        'eda_scr_amplitude_mean', 'eda_scr_amplitude_std',
        'eda_scr_rise_time_mean', 'eda_scr_rise_time_std', 'eda_scr_per_min',
    ],
    'respiration': [
        'respiration_breath_rate_mean', 'respiration_breath_rate_std',
        'respiration_inspiratory_time_mean', 'respiration_inspiratory_time_std',
        'respiration_expiratory_time_mean', 'respiration_expiratory_time_std',
        'respiration_timing_ratio_mean', 'respiration_timing_ratio_std',
        'respiration_amplitude_mean', 'respiration_amplitude_std',
        'respiration_minute_ventilation_mean', 'respiration_minute_ventilation_std',
        'respiration_coefficient_of_variation', 'respiration_autocorrelation',
    ],
    'temperature': [
        'temperature_delta', 'temperature_mean', 'temperature_std',
        'temperature_delta_over_time', 'temperature_linear_slope',
        'temperature_derivative_mean', 'temperature_derivative_std',
        'temperature_derivative_linear_slope',
    ],
    'fnirs': (
        [f'fnirs_oxy_{name}' for name in fnirs_suffixes]
        + [f'fnirs_deoxy_{name}' for name in fnirs_suffixes]
    ),
}

assert {name: len(values) for name, values in expected_features.items()} == {
    'ecg': 11,
    'eda': 8,
    'respiration': 14,
    'temperature': 8,
    'fnirs': 38,
}


# Cross-language tolerances for continuous features.
#
# These tolerances account for expected numerical differences between
# MATLAB and Python implementations (e.g. filtering and numerical solvers),
# while discrete physiological outcomes should remain equivalent.
tolerances = {
    'ecg': {
        'rtol': 1e-6,
        'atol': 1e-9,
    },
    'eda': {
        'rtol': 3e-2,
        'atol': 1e-6,
    },
    'respiration': {
        'rtol': 3e-2,
        'atol': 1e-6,
    },
    'temperature': {
        'rtol': 3e-2,
        'atol': 1e-6,
    },
    'fnirs': {
        'rtol': 1e-6,
        'atol': 1e-9,
    },
}


# Features representing discrete event populations should not be allowed
# to pass through a relaxed relative tolerance.
exact_features = {
    'eda_scr_per_min',
}

# Rise-time quantities are measured in seconds.
# At 100 Hz, 0.01 s corresponds to one sample, so allow up to
# 0.02 s absolute cross-language variation.
eda_rise_time_features = {
    'eda_scr_rise_time_mean',
    'eda_scr_rise_time_std',
}

EDA_RISE_TIME_ATOL = 0.02

def compare_modality(modality):
    rows = []
    feature_names = expected_features[modality]

    assert set(python_features[modality]) == set(phase_names)
    assert set(matlab_reference[modality]) == set(phase_names)

    for phase in phase_names:
        python_phase = python_features[modality][phase]
        matlab_phase = matlab_reference[modality][phase]

        assert set(python_phase) == set(feature_names)
        assert set(matlab_phase) == set(feature_names)

        for feature in feature_names:
            python_value = float(python_phase[feature])
            matlab_value = float(matlab_phase[feature])

            finite = (
                np.isfinite(python_value)
                and np.isfinite(matlab_value)
            )

            absolute_difference = abs(
                python_value - matlab_value
            )

            scale = max(
                abs(python_value),
                abs(matlab_value),
            )

            relative_difference = (
                absolute_difference / scale
                if scale > 0
                else 0.0
            )

            if feature in exact_features:
                passed = bool(
                    finite
                    and np.isclose(
                        python_value,
                        matlab_value,
                        rtol=0.0,
                        atol=1e-12,
                    )
                )

            elif modality == 'eda' and feature in eda_rise_time_features:
                passed = bool(
                    finite
                    and np.isclose(
                        python_value,
                        matlab_value,
                        rtol=tolerances['eda']['rtol'],
                        atol=EDA_RISE_TIME_ATOL,
                    )
                )

            else:
                passed = bool(
                    finite
                    and np.isclose(
                        python_value,
                        matlab_value,
                        **tolerances[modality],
                    )
                )

            rows.append({
                'Phase': phase,
                'Feature': feature,
                'Python': python_value,
                'MATLAB': matlab_value,
                'Absolute difference': absolute_difference,
                'Relative difference': relative_difference,
                'Pass': passed,
            })

    return pd.DataFrame(rows)


def summarize_modality(comparison):
    comparisons = len(comparison)
    passed = int(comparison['Pass'].sum())
    failed = comparisons - passed

    return {
        'Comparisons': comparisons,
        'Passed': passed,
        'Failed': failed,
        'Max absolute difference': comparison['Absolute difference'].max(),
        'Max relative difference': comparison['Relative difference'].max(),
        'Status': 'PASS' if failed == 0 else 'FAIL',
    }


def report_mismatches(label, comparison):
    mismatches = comparison.loc[~comparison['Pass']].copy()

    if mismatches.empty:
        print(f'{label}: all features match within tolerance.')
    else:
        mismatches = mismatches.sort_values(
            'Relative difference',
            ascending=False,
        )

        print(
            f'{label}: {len(mismatches)} feature comparison(s) '
            'exceed tolerance.'
        )
        display(mismatches)

    return mismatches

## ECG validation

Compare all 11 ECG final features across the three common phases.

In [ ]:
ecg_comparison = compare_modality('ecg')
ecg_comparison

In [ ]:
ecg_mismatches = report_mismatches('ECG', ecg_comparison)

In [ ]:
ecg_summary = summarize_modality(ecg_comparison)
pd.Series(ecg_summary, name='ECG validation summary')

## EDA validation

Compare the eight final EDA features produced by the complete real cvxEDA v1.1.0 workflow. Accepted event counts (the discrete basis of SCR/min) are checked exactly when the MATLAB reference is loaded; continuous features use the documented cross-language tolerance, with a 0.02 s absolute allowance for rise-time features.

In [ ]:
eda_comparison = compare_modality('eda')
eda_comparison

In [ ]:
eda_mismatches = report_mismatches('EDA', eda_comparison)

In [ ]:
eda_summary = summarize_modality(eda_comparison)
pd.Series(eda_summary, name='EDA validation summary')

## Respiration validation

Compare all 14 respiration final features phase by phase.

In [ ]:
respiration_comparison = compare_modality('respiration')
respiration_comparison

In [ ]:
respiration_mismatches = report_mismatches('Respiration', respiration_comparison)

In [ ]:
respiration_summary = summarize_modality(respiration_comparison)
pd.Series(respiration_summary, name='Respiration validation summary')

## Temperature validation

Compare all eight temperature final features across the common phases.

In [ ]:
temperature_comparison = compare_modality('temperature')
temperature_comparison

In [ ]:
temperature_mismatches = report_mismatches('Temperature', temperature_comparison)

In [ ]:
temperature_summary = summarize_modality(temperature_comparison)
pd.Series(temperature_summary, name='Temperature validation summary')

## fNIRS validation

Compare all 38 fNIRS final features per phase. Failures are sorted by descending symmetric relative difference for easier inspection. Oxy and Deoxy rows can be inspected with ordinary DataFrame filtering.

In [ ]:
fnirs_comparison = compare_modality('fnirs')
fnirs_comparison

In [ ]:
fnirs_mismatches = (
    fnirs_comparison.loc[~fnirs_comparison['Pass']]
    .sort_values('Relative difference', ascending=False)
)
if fnirs_mismatches.empty:
    print('fNIRS: all features match within tolerance.')
else:
    print(f'fNIRS: {len(fnirs_mismatches)} feature comparisons exceed tolerance.')
    display(fnirs_mismatches)

In [ ]:
for row in fnirs_mismatches.itertuples(index=False):
    print(np.array(row))

In [ ]:
fnirs_summary = summarize_modality(fnirs_comparison)
pd.Series(fnirs_summary, name='fNIRS validation summary')

## Overall validation summary

The compact summary appears only after all five modality-specific reports. In total, 79 feature definitions produce 237 phase-level comparisons.

In [ ]:
modality_summaries = {
    'ECG': ecg_summary,
    'EDA': eda_summary,
    'Respiration': respiration_summary,
    'Temperature': temperature_summary,
    'fNIRS': fnirs_summary,
}
overall_summary = pd.DataFrame([
    {'Modality': modality, **summary}
    for modality, summary in modality_summaries.items()
])
assert list(overall_summary['Comparisons']) == [33, 24, 42, 24, 114]
overall_summary

In [ ]:
failed_modalities = overall_summary.loc[overall_summary['Status'] == 'FAIL', 'Modality'].tolist()
if not failed_modalities:
    print('MATLAB/Python cross-language validation: PASS')
else:
    print('MATLAB/Python cross-language validation: FAIL')
    print('Failed modalities:', ', '.join(failed_modalities))